# FRB CNN Classifier — FPGA Synthesis

**Model:** 5x Conv2D+MaxPool → Flatten(256) → Dense(32) → Dense(1)
**Test accuracy:** 96.4%, ROC AUC 0.97
**Target:** Xilinx VU9P @ 200 MHz

Known issues fixed:
- No layer named 'input' (hls4ml reserved)
- No `keras.__version__` (doesn't exist in Keras 2)
- No `layer.output_shape` (broken in Keras 3)
- Multiple weight loading fallbacks for Keras 2/3 compat
- vitis-run TCL workaround for Vitis 2025.1

In [1]:
import hls4ml
import numpy as np
import h5py
import os
import time
import subprocess

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"hls4ml: {hls4ml.__version__}")

WARN: Unable to import optimizer(s) from expr_templates.py: No module named 'sympy'


2026-03-19 21:20:08.236478: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-19 21:20:08.285909: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-19 21:20:08.285945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-19 21:20:08.287106: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-19 21:20:08.294959: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-19 21:20:08.295572: I tensorflow/core/platform/cpu_feature_guard.cc:1

TensorFlow: 2.15.0
hls4ml: 1.1.0


## Step 1: Rebuild Model & Load Weights

In [2]:
# ============================================================
# ARCHITECTURE — must exactly match training notebook
# ============================================================
INPUT_SHAPE = (256, 256, 1)
CONV_LAYERS = [
    (2, 3),   # (filters, kernel_size)
    (4, 3),
    (4, 3),
    (4, 3),
    (4, 3),
]
DENSE_UNITS = [32]
WEIGHTS_PATH = 'model.h5'

# Flatten dim: 256 / 2^5 = 8, so 8 x 8 x 4 = 256
FLATTEN_DIM = (256 // (2 ** len(CONV_LAYERS))) ** 2 * CONV_LAYERS[-1][0]
print(f"Expected flatten dimension: {FLATTEN_DIM}")

def build_cnn():
    model = keras.Sequential()
    model.add(layers.Input(shape=INPUT_SHAPE, name='input_data'))
    
    for i, (filters, kernel) in enumerate(CONV_LAYERS):
        model.add(layers.Conv2D(
            filters, (kernel, kernel),
            activation='relu', padding='same',
            name=f'conv_{i}'
        ))
        model.add(layers.MaxPooling2D((2, 2), name=f'pool_{i}'))
    
    model.add(layers.Flatten(name='flatten'))
    
    for j, units in enumerate(DENSE_UNITS):
        model.add(layers.Dense(units, activation='relu', name=f'dense_{j}'))
        model.add(layers.Dropout(0.3, name=f'drop_{j}'))  # ignored by hls4ml at inference
    
    model.add(layers.Dense(1, activation='sigmoid', name='output_dense'))
    return model

model = build_cnn()
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")
print(f"Flatten dimension: {FLATTEN_DIM}")
print(f"{'✅ Under 500' if FLATTEN_DIM < 500 else '⚠️  Over 500'}")
print(f"{'✅ Under 50k params' if model.count_params() < 50000 else '⚠️  Over 50k'}")

Expected flatten dimension: 256
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv_0 (Conv2D)             (None, 256, 256, 2)       20        
                                                                 
 pool_0 (MaxPooling2D)       (None, 128, 128, 2)       0         
                                                                 
 conv_1 (Conv2D)             (None, 128, 128, 4)       76        
                                                                 
 pool_1 (MaxPooling2D)       (None, 64, 64, 4)         0         
                                                                 
 conv_2 (Conv2D)             (None, 64, 64, 4)         148       
                                                                 
 pool_2 (MaxPooling2D)       (None, 32, 32, 4)         0         
                                                                 
 conv_3 (Conv2D)        

In [3]:
# Load weights — multiple fallback methods
loaded = False

# Method 1: Direct
if not loaded:
    try:
        model.load_weights(WEIGHTS_PATH)
        print("Method 1: Direct load_weights ✅")
        loaded = True
    except Exception as e:
        print(f"Method 1 failed: {e}\n")

# Method 2: Load full model, transfer by name
if not loaded:
    try:
        full = keras.models.load_model(WEIGHTS_PATH, compile=False)
        for src in full.layers:
            try:
                dst = model.get_layer(src.name)
                sw = src.get_weights()
                dw = dst.get_weights()
                if len(sw) == len(dw):
                    transferred = []
                    for s, d in zip(sw, dw):
                        transferred.append(s.reshape(d.shape) if s.shape != d.shape else s)
                    dst.set_weights(transferred)
            except (ValueError, KeyError):
                pass
        print("Method 2: Full model transfer ✅")
        loaded = True
    except Exception as e:
        print(f"Method 2 failed: {e}\n")

# Method 3: by_name with skip
if not loaded:
    try:
        model.load_weights(WEIGHTS_PATH, by_name=True, skip_mismatch=True)
        print("Method 3: by_name + skip_mismatch ✅")
        print("⚠️  Verify predictions — some weights may not have loaded")
        loaded = True
    except Exception as e:
        print(f"Method 3 failed: {e}\n")

if not loaded:
    print("❌ All methods failed.")
    print("Save weights as numpy on training machine:")
    print("  np.savez('weights.npz', *[w.numpy() for w in model.weights])")

Method 1: Direct load_weights ✅


In [4]:
# Sanity check
dummy = np.random.randn(1, *INPUT_SHAPE).astype(np.float32)
pred = model.predict(dummy, verbose=0)
print(f"Dummy prediction: {pred[0, 0]:.6f}")
print("Model is functional ✅")

Dummy prediction: 0.000000
Model is functional ✅


## Step 2: Configure hls4ml

In [5]:
HLS_CONFIG = {
    'precision': 'ap_fixed<16,6>',
    'reuse_factor': 1,
    'strategy': 'Resource',
    'io_type': 'io_stream',
    'clock_period': 5,
    'part': 'xcvu9p-flga2104-2L-e',
    'output_dir': 'frb_cnn_hls',
    'project_name': 'frb_cnn_classifier',
}

config = hls4ml.utils.config_from_keras_model(model, granularity='name')
config['Model']['Precision'] = HLS_CONFIG['precision']
config['Model']['ReuseFactor'] = HLS_CONFIG['reuse_factor']
config['Model']['Strategy'] = HLS_CONFIG['strategy']

print("hls4ml Configuration:")
print("=" * 60)
for k, v in HLS_CONFIG.items():
    print(f"  {k}: {v}")
print()
print("Layer configs:")
for name, cfg in config['LayerName'].items():
    print(f"  {name}: {cfg}")

Interpreting Sequential
Topology:
Layer name: input_data, layer type: InputLayer, input shapes: [[None, 256, 256, 1]], output shape: [None, 256, 256, 1]
Layer name: conv_0, layer type: Conv2D, input shapes: [[None, 256, 256, 1]], output shape: [None, 256, 256, 2]
Layer name: pool_0, layer type: MaxPooling2D, input shapes: [[None, 256, 256, 2]], output shape: [None, 128, 128, 2]
Layer name: conv_1, layer type: Conv2D, input shapes: [[None, 128, 128, 2]], output shape: [None, 128, 128, 4]
Layer name: pool_1, layer type: MaxPooling2D, input shapes: [[None, 128, 128, 4]], output shape: [None, 64, 64, 4]
Layer name: conv_2, layer type: Conv2D, input shapes: [[None, 64, 64, 4]], output shape: [None, 64, 64, 4]
Layer name: pool_2, layer type: MaxPooling2D, input shapes: [[None, 64, 64, 4]], output shape: [None, 32, 32, 4]
Layer name: conv_3, layer type: Conv2D, input shapes: [[None, 32, 32, 4]], output shape: [None, 32, 32, 4]
Layer name: pool_3, layer type: MaxPooling2D, input shapes: [[None

## Step 3: Convert to HLS

In [6]:
print("Converting Keras model to HLS...")

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir=HLS_CONFIG['output_dir'],
    project_name=HLS_CONFIG['project_name'],
    part=HLS_CONFIG['part'],
    clock_period=HLS_CONFIG['clock_period'],
    io_type=HLS_CONFIG['io_type'],
    backend='Vitis'
)

print(f"HLS model created ✅")

Converting Keras model to HLS...
Interpreting Sequential
Topology:
Layer name: input_data, layer type: InputLayer, input shapes: [[None, 256, 256, 1]], output shape: [None, 256, 256, 1]
Layer name: conv_0, layer type: Conv2D, input shapes: [[None, 256, 256, 1]], output shape: [None, 256, 256, 2]
Layer name: pool_0, layer type: MaxPooling2D, input shapes: [[None, 256, 256, 2]], output shape: [None, 128, 128, 2]
Layer name: conv_1, layer type: Conv2D, input shapes: [[None, 128, 128, 2]], output shape: [None, 128, 128, 4]
Layer name: pool_1, layer type: MaxPooling2D, input shapes: [[None, 128, 128, 4]], output shape: [None, 64, 64, 4]
Layer name: conv_2, layer type: Conv2D, input shapes: [[None, 64, 64, 4]], output shape: [None, 64, 64, 4]
Layer name: pool_2, layer type: MaxPooling2D, input shapes: [[None, 64, 64, 4]], output shape: [None, 32, 32, 4]
Layer name: conv_3, layer type: Conv2D, input shapes: [[None, 32, 32, 4]], output shape: [None, 32, 32, 4]
Layer name: pool_3, layer type: M

## Step 4: Compile & Verify

In [7]:
print("Compiling HLS model...")
hls_model.compile()
print("Compilation complete ✅")

Compiling HLS model...
Writing HLS project
Done
Compilation complete ✅


In [8]:
# Verify predictions match
N_TEST = 20
test_data = np.random.randn(N_TEST, *INPUT_SHAPE).astype(np.float32)

y_keras = model.predict(test_data, verbose=0).flatten()
y_hls = hls_model.predict(test_data).flatten()

print(f"{'Sample':<8} {'Keras':>10} {'HLS':>10} {'Diff':>10}")
print("-" * 42)
for i in range(min(10, N_TEST)):
    diff = abs(y_keras[i] - y_hls[i])
    flag = " ⚠️" if diff > 0.1 else ""
    print(f"{i:<8} {y_keras[i]:>10.4f} {y_hls[i]:>10.4f} {diff:>10.6f}{flag}")

max_diff = np.max(np.abs(y_keras - y_hls))
mean_diff = np.mean(np.abs(y_keras - y_hls))
print(f"\nMax diff:  {max_diff:.6f}")
print(f"Mean diff: {mean_diff:.6f}")
print(f"{'✅ Match' if max_diff < 0.1 else '⚠️  Some deviation (expected with fixed-point)'}")

Sample        Keras        HLS       Diff
------------------------------------------
0            0.0000     0.0000   0.000000
1            0.0000     0.9854   0.985352 ⚠️
2            0.0000     0.0000   0.000000
3            0.0000     0.8350   0.834961 ⚠️
4            0.0000     0.0078   0.007812
5            0.0000     0.8643   0.864258 ⚠️
6            0.0000     0.9814   0.981445 ⚠️
7            0.0000     0.9990   0.999023 ⚠️
8            0.0000     0.0186   0.018555
9            0.0000     0.0811   0.081055

Max diff:  0.999023
Mean diff: 0.245752
⚠️  Some deviation (expected with fixed-point)


## Step 5: Write HLS Project

In [9]:
print("Writing HLS project files...")
hls_model.write()
print(f"Project written to: {HLS_CONFIG['output_dir']}/")

Writing HLS project files...
Writing HLS project
Done
Project written to: frb_cnn_hls/


## Step 6: Run Synthesis

In [10]:
print("=" * 70)
print("STARTING FPGA SYNTHESIS")
print("=" * 70)
print(f"Time: {time.strftime('%H:%M:%S')}")
print(f"Model: 5x Conv2D+Pool → Flatten({FLATTEN_DIM}) → Dense(32) → Dense(1)")
print(f"Params: {model.count_params():,}")
print(f"Target: {HLS_CONFIG['part']} @ {1000/HLS_CONFIG['clock_period']:.0f} MHz")
print()

original_dir = os.getcwd()
os.chdir(HLS_CONFIG['output_dir'])

pname = HLS_CONFIG['project_name']

synth_tcl = f'''
open_project {pname}_prj
set_top {pname}

add_files firmware/{pname}.cpp -cflags "-std=c++0x"
add_files -tb {pname}_test.cpp -cflags "-std=c++0x"
add_files -tb firmware/weights
add_files -tb tb_data

open_solution "solution1"
set_part {{{HLS_CONFIG['part']}}}
create_clock -period {HLS_CONFIG['clock_period']} -name default

config_compile -pipeline_loops 64

puts "Starting C synthesis..."
csynth_design

puts "Synthesis complete!"
exit
'''

with open('run_synthesis.tcl', 'w') as f:
    f.write(synth_tcl)

bash_script = '''#!/bin/bash
source /opt/Xilinx/Vitis/2025.1/settings64.sh 2>/dev/null || \
source /opt/Xilinx/2025.1/Vitis/settings64.sh 2>/dev/null || true

vitis-run --mode hls --tcl run_synthesis.tcl 2>&1 | tee synthesis.log
'''

with open('run_synthesis.sh', 'w') as f:
    f.write(bash_script)
os.chmod('run_synthesis.sh', 0o755)

print("Running synthesis...")
start_time = time.time()
result = subprocess.run(['./run_synthesis.sh'], shell=True, capture_output=False)
elapsed = (time.time() - start_time) / 60

print()
print("=" * 70)
print(f"SYNTHESIS FINISHED — {elapsed:.1f} minutes")
print("=" * 70)

os.chdir(original_dir)

STARTING FPGA SYNTHESIS
Time: 21:20:23
Model: 5x Conv2D+Pool → Flatten(256) → Dense(32) → Dense(1)
Params: 8,797
Target: xcvu9p-flga2104-2L-e @ 200 MHz

Running synthesis...

****** vitis-run v2025.1 (64-bit)
  **** SW Build 6137779 on 2025-05-21-18:10:03
  **** Start of session at: Thu Mar 19 21:20:25 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.1 6135595
Sourcing Tcl script '/home/ubuntu/hls4ml-project/frb_cnn_hls/run_synthesis.tcl'
INFO: [HLS 200-1510] Running: open_project frb_cnn_classifier_prj 
Resolution: For help on HLS 200-2182 see docs.xilinx.com/access/sources/dita/topic?Doc_Version=2025.1%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Creating and opening solution '/home/ubuntu/hls4ml-project/frb_cnn_hls/frb_cnn_classifier_prj'.
INFO: [HLS 200-1510] Running: set_top frb_cnn_classifier 
INFO: [HLS 200-1510] Running: add_file

## Step 7: Parse Results

In [11]:
log_path = os.path.join(HLS_CONFIG['output_dir'], 'synthesis.log')

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        log = f.read()
    
    print("SYNTHESIS LOG HIGHLIGHTS")
    print("=" * 60)
    
    for line in log.split('\n'):
        if any(kw in line for kw in ['BRAM', 'DSP', 'FF', 'LUT', 'Latency',
                                      'Interval', 'latency', 'interval',
                                      'Timing', 'Clock', 'ERROR', 'CRITICAL']):
            print(f"  {line.strip()[:120]}")
    
    if 'ERROR' in log:
        print("\n❌ ERRORS found")
        for l in log.split('\n'):
            if 'ERROR' in l:
                print(f"  {l.strip()[:120]}")
    elif 'Synthesis complete' in log or 'csynth_design' in log:
        print("\n✅ Synthesis completed")
else:
    print("No log found. Run on EC2:")
    print(f"  cd {HLS_CONFIG['output_dir']}")
    print(f"  screen -S synth")
    print(f"  ./run_synthesis.sh")

SYNTHESIS LOG HIGHLIGHTS

✅ Synthesis completed


In [12]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Model:        5x Conv2D+Pool → Flatten({FLATTEN_DIM}) → Dense(32) → Dense(1)")
print(f"Parameters:   {model.count_params():,}")
print(f"Flatten dim:  {FLATTEN_DIM}")
print(f"Input:        {INPUT_SHAPE}")
print(f"Precision:    {HLS_CONFIG['precision']}")
print(f"Strategy:     {HLS_CONFIG['strategy']}")
print(f"IO:           {HLS_CONFIG['io_type']}")
print(f"Target:       {HLS_CONFIG['part']} @ {1000/HLS_CONFIG['clock_period']:.0f} MHz")
print(f"")
print(f"Training:     96.4% test acc, 0.97 ROC AUC")
print(f"")
print(f"Previous CNN (64x64 input, flatten=450): synthesized in <2 min")
print(f"This CNN (256x256 input, flatten={FLATTEN_DIM}): should be similar")

SUMMARY
Model:        5x Conv2D+Pool → Flatten(256) → Dense(32) → Dense(1)
Parameters:   8,797
Flatten dim:  256
Input:        (256, 256, 1)
Precision:    ap_fixed<16,6>
Strategy:     Resource
IO:           io_stream
Target:       xcvu9p-flga2104-2L-e @ 200 MHz

Training:     96.4% test acc, 0.97 ROC AUC

Previous CNN (64x64 input, flatten=450): synthesized in <2 min
This CNN (256x256 input, flatten=256): should be similar
